# NumPy — Numerical Computing with Python

## What is NumPy and why use it?
NumPy (Numerical Python) provides the `ndarray` — a C-backed, homogeneous array that is orders of magnitude faster than a Python list for numerical work. Nearly every scientific Python library (pandas, scikit-learn, TensorFlow) is built on top of NumPy arrays.

**Key advantages:**
- Vectorised operations eliminate explicit Python loops
- Contiguous memory layout means CPU cache friendliness
- Broadcasting lets you operate on arrays of different shapes without copies

```bash
pip install numpy
```

## Table of Contents
1. `ndarray` vs Python list
2. Creating arrays
3. Array attributes: shape, dtype, ndim, size
4. Indexing and slicing
5. Reshaping arrays
6. Broadcasting
7. Universal functions (ufuncs)
8. Aggregations
9. Matrix operations
10. Stacking arrays
11. Saving and loading
12. Common Mistakes
13. Exercises + Solutions
14. Mini-Project: Statistics Calculator

---
## 1. `ndarray` vs Python list
NumPy arrays store data in a single contiguous block of typed memory. Python lists store pointers to arbitrary objects. The difference is dramatic for numeric work.

In [ ]:
import numpy as np
import time

size = 1_000_000

# Python list
py_list = list(range(size))
start = time.perf_counter()
py_result = [x * 2 for x in py_list]
py_time = time.perf_counter() - start

# NumPy array
np_arr = np.arange(size)
start = time.perf_counter()
np_result = np_arr * 2
np_time = time.perf_counter() - start

print(f"Python list : {py_time*1000:.2f} ms")
print(f"NumPy array : {np_time*1000:.2f} ms")
print(f"NumPy is ~{py_time/np_time:.0f}x faster")

---
## 2. Creating Arrays

In [ ]:
import numpy as np

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("array:", a)

# Filled arrays
print("zeros:", np.zeros((2, 3)))
print("ones: ", np.ones((2, 3)))
print("full: ", np.full((2, 3), 7))

# Ranges
print("arange(0,10,2):", np.arange(0, 10, 2))  # like range()
print("linspace(0,1,5):", np.linspace(0, 1, 5))  # N evenly-spaced points

# Random arrays
rng = np.random.default_rng(seed=42)  # reproducible
print("uniform [0,1):", rng.random((2, 3)))
print("integers:     ", rng.integers(0, 10, size=(2, 4)))
print("normal:       ", rng.standard_normal((2, 3)).round(2))

---
## 3. Array Attributes

In [ ]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)

print("shape:", arr.shape)   # (rows, cols)
print("ndim: ", arr.ndim)    # number of dimensions
print("size: ", arr.size)    # total number of elements
print("dtype:", arr.dtype)   # element type
print("itemsize:", arr.itemsize, "bytes")  # bytes per element
print("nbytes:", arr.nbytes, "bytes")      # total memory

# Change dtype
int_arr = arr.astype(np.int64)
print("\nConverted to int64:", int_arr.dtype)

---
## 4. Indexing and Slicing

In [ ]:
import numpy as np

# 1D indexing — same as Python lists
a = np.array([10, 20, 30, 40, 50])
print("a[0]:", a[0])       # 10
print("a[-1]:", a[-1])     # 50
print("a[1:4]:", a[1:4])   # [20 30 40]
print("a[::2]:", a[::2])   # [10 30 50]

# 2D indexing — [row, col]
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print("\nm[1, 2]:", m[1, 2])           # 6
print("Row 0:  ", m[0, :])             # [1 2 3]
print("Col 1:  ", m[:, 1])             # [2 5 8]
print("Sub:    ", m[:2, :2])           # top-left 2x2

# WARNING: slices are views, not copies!
view = m[:2, :2]
view[0, 0] = 99
print("m after view mutation:", m[0, 0])  # 99!

In [ ]:
import numpy as np

# Boolean indexing — filter elements by condition
scores = np.array([72, 85, 63, 91, 78, 55, 88])
passing = scores[scores >= 70]
print("Passing scores:", passing)

# Fancy indexing — select by index array
selected = scores[[0, 2, 5]]
print("Selected indices:", selected)

# where — conditional replacement
grade = np.where(scores >= 70, "pass", "fail")
print("Grades:", grade)

---
## 5. Reshaping Arrays

In [ ]:
import numpy as np

a = np.arange(12)
print("Original:", a)

# reshape — gives a view with new shape (must keep same number of elements)
m = a.reshape(3, 4)
print("\nreshaped to (3,4):\n", m)

# Use -1 to let NumPy infer one dimension
m2 = a.reshape(2, -1)  # 2 rows, auto cols
print("\nreshaped to (2,-1):\n", m2)

# flatten — always returns a copy
flat = m.flatten()
print("\nflattened:", flat)

# ravel — returns a view when possible
raveled = m.ravel()
print("raveled:  ", raveled)

# Add a new axis
v = np.array([1, 2, 3])
print("\nOriginal shape:", v.shape)
print("Column vector:  ", v[:, np.newaxis].shape)  # (3,1)
print("Row vector:     ", v[np.newaxis, :].shape)  # (1,3)

---
## 6. Broadcasting
Broadcasting is how NumPy handles arithmetic between arrays of different shapes. The key rule: dimensions are compared from the right; a dimension of 1 can be stretched to match the other.

In [ ]:
import numpy as np

# Scalar broadcast
a = np.array([1, 2, 3, 4])
print("a * 2:", a * 2)   # [2 4 6 8] — scalar stretches to match

# 1D broadcast over 2D
matrix = np.array([[1, 2, 3],   # shape (2,3)
                   [4, 5, 6]])
row    = np.array([10, 20, 30])  # shape   (3,)
print("matrix + row:\n", matrix + row)  # row is broadcast across both rows

# Column vector broadcast
col = np.array([[100],   # shape (2,1)
                [200]])
print("matrix + col:\n", matrix + col)  # col broadcast across all columns

# Practical: normalise each row
row_means = matrix.mean(axis=1, keepdims=True)  # shape (2,1)
normalised = matrix - row_means
print("Row-normalised:\n", normalised)

---
## 7. Universal Functions (ufuncs)
Ufuncs are vectorised wrappers around fast C operations — they replace loops and work element-wise on arrays.

In [ ]:
import numpy as np

x = np.array([0, 1, 2, 3, 4], dtype=float)

print("sqrt:  ", np.sqrt(x))
print("exp:   ", np.exp(x).round(2))
print("log:   ", np.log(x + 1).round(3))   # log(0) is -inf, add 1
print("sin:   ", np.sin(x * np.pi / 4).round(3))
print("abs:   ", np.abs(np.array([-3, -1, 0, 2, 4])))

a = np.array([1, 5, 3])
b = np.array([4, 2, 6])
print("maximum:", np.maximum(a, b))  # element-wise max
print("add:    ", np.add(a, b))      # same as a + b

---
## 8. Aggregations
Aggregation functions reduce an array along an axis. The `axis` parameter controls *which* dimension collapses.

In [ ]:
import numpy as np

# Student scores: rows=students, cols=subjects
scores = np.array([
    [85, 92, 78, 95],   # student 0
    [70, 65, 88, 72],   # student 1
    [95, 98, 91, 97],   # student 2
])

print("Shape:", scores.shape)  # (3, 4)

# Overall
print("\nOverall mean: ", scores.mean())
print("Overall std:  ", scores.std().round(2))

# axis=1 → collapse columns → result per row (per student)
print("\nPer-student average (axis=1):", scores.mean(axis=1))
print("Per-student max    (axis=1):", scores.max(axis=1))

# axis=0 → collapse rows → result per column (per subject)
print("\nPer-subject average (axis=0):", scores.mean(axis=0))
print("Per-subject min     (axis=0):", scores.min(axis=0))

# argmax — index of maximum value
top_student = scores.mean(axis=1).argmax()
print(f"\nTop student: index {top_student} with avg {scores[top_student].mean():.1f}")

---
## 9. Matrix Operations

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Element-wise multiplication (NOT matrix multiply)
print("Element-wise A*B:\n", A * B)

# Matrix multiplication — three equivalent ways
print("\nMatrix multiply A@B:\n",    A @ B)          # preferred
print("np.matmul(A,B):\n",           np.matmul(A, B))
print("np.dot(A,B):\n",              np.dot(A, B))

# Transpose
print("\nTranspose A.T:\n", A.T)

# Linear algebra
print("\nDeterminant:", np.linalg.det(A))
print("Inverse:\n",     np.linalg.inv(A).round(3))

# Solve Ax = b
b = np.array([5, 11])
x = np.linalg.solve(A, b)
print("Solution x:", x)  # [1, 2]

---
## 10. Stacking Arrays

In [ ]:
import numpy as np

a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

# vstack — stack vertically (add rows)
print("vstack:\n", np.vstack([a, b]))

# hstack — stack horizontally (add columns)
print("hstack:\n", np.hstack([a, b]))

# concatenate — generalised stacking along any axis
print("concatenate axis=0:\n", np.concatenate([a, b], axis=0))  # same as vstack
print("concatenate axis=1:\n", np.concatenate([a, b], axis=1))  # same as hstack

# stack — add a new dimension
stacked = np.stack([a, b], axis=0)  # shape becomes (2, 2, 2)
print("\nstack shape:", stacked.shape)

---
## 11. Saving and Loading

In [ ]:
import numpy as np
from pathlib import Path

Path("temp").mkdir(exist_ok=True)

data = np.random.default_rng(0).random((4, 3))

# Binary format (fast, lossless, preserves dtype)
np.save("temp/data.npy", data)
loaded = np.load("temp/data.npy")
print("Loaded .npy:", loaded.shape, loaded.dtype)

# Multiple arrays in one file
np.savez("temp/data.npz", features=data, labels=np.array([0, 1, 1, 0]))
bundle = np.load("temp/data.npz")
print("Keys:", list(bundle.keys()))

# Text format — readable, slower
np.savetxt("temp/data.csv", data, delimiter=",", header="a,b,c", comments="")
loaded_csv = np.loadtxt("temp/data.csv", delimiter=",", skiprows=1)
print("Loaded .csv:", loaded_csv.shape)

---
## 12. Common Mistakes

**Slices are views:** `a[1:3]` does *not* copy data. Modifying the slice modifies the original. Use `.copy()` if you need independence.

**`*` vs `@`:** `A * B` is element-wise multiplication; `A @ B` is matrix multiplication. Mixing them up produces wrong results silently.

**Shape `(n,)` vs `(n,1)` vs `(1,n)`:** These behave differently in broadcasting. Use `reshape(-1, 1)` or `[:, np.newaxis]` to add an axis explicitly.

**Integer division vs float:** `np.array([1, 2, 3]) / 2` gives floats. `np.array([1, 2, 3]) // 2` gives integers. If your array dtype is `int64`, arithmetic stays integer until you explicitly convert.

**`np.random.seed()` is legacy:** Use `np.random.default_rng(seed)` for modern, reproducible random number generation.

---
## 13. Exercises

**Exercise 1:** Create a 5x5 matrix of random integers between 1 and 100. Replace every value above 50 with 1 and every value 50 or below with 0. Print the result.

**Exercise 2:** Given a 1D array of 20 random values, reshape it to (4, 5), then compute the column-wise cumulative sum.

**Exercise 3:** Create two arrays `a = [1,2,3]` and `b = [4,5,6]`. Compute their dot product, outer product, and element-wise sum without using a loop.

In [ ]:
# Solution 1
import numpy as np
rng = np.random.default_rng(42)
m = rng.integers(1, 101, size=(5, 5))
print("Original:\n", m)
binary = (m > 50).astype(int)
print("Binary:\n", binary)

In [ ]:
# Solution 2
import numpy as np
rng = np.random.default_rng(0)
arr = rng.integers(1, 10, size=20)
m = arr.reshape(4, 5)
print("Matrix:\n", m)
print("Column cumsum:\n", m.cumsum(axis=0))

In [ ]:
# Solution 3
import numpy as np
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("Dot product:   ", np.dot(a, b))
print("Outer product:\n", np.outer(a, b))
print("Element-wise + :", a + b)

---
## 14. Mini-Project: Statistics Calculator
Given a 2D array of student scores (rows = students, cols = subjects), compute per-student averages, per-subject averages, find the top student, and normalise all scores to the 0-1 range.

In [ ]:
import numpy as np

# Simulate score data: 8 students, 5 subjects
rng = np.random.default_rng(99)
scores = rng.integers(40, 100, size=(8, 5)).astype(float)
students = [f"Student {i+1}" for i in range(8)]
subjects = ["Math", "English", "Science", "History", "Art"]

print("Raw scores (students x subjects):")
header = f"{'':12s}" + "".join(f"{s:10s}" for s in subjects)
print(header)
for name, row in zip(students, scores):
    print(f"{name:12s}" + "".join(f"{v:10.0f}" for v in row))

# Per-student average
student_avg = scores.mean(axis=1)
print("\nPer-student averages:")
for name, avg in zip(students, student_avg):
    print(f"  {name}: {avg:.1f}")

# Per-subject average
subject_avg = scores.mean(axis=0)
print("\nPer-subject averages:")
for subj, avg in zip(subjects, subject_avg):
    print(f"  {subj}: {avg:.1f}")

# Top student
top_idx = student_avg.argmax()
print(f"\nTop student: {students[top_idx]} with average {student_avg[top_idx]:.1f}")

# Normalise to 0-1 (global min-max)
s_min, s_max = scores.min(), scores.max()
normalised = (scores - s_min) / (s_max - s_min)
print(f"\nNormalised scores (range: {normalised.min():.2f} to {normalised.max():.2f}):")
print(normalised.round(3))

# Rank students by average
ranked_idx = student_avg.argsort()[::-1]
print("\nRankings:")
for rank, idx in enumerate(ranked_idx, 1):
    print(f"  {rank}. {students[idx]:12s} {student_avg[idx]:.1f}")